# Neural Networks for Unsupervised Learning -- Autoencoders

In this exercise we build an **autoencoder** -- a neural network that learns to compress data into a low-dimensional latent representation and then reconstruct it.

**Dataset:** `classif_train.csv` (10,000 samples) and `classif_test.csv` (1,000 samples) with columns `x`, `y`, and `label`. The points are spread uniformly over a square, and `label` marks the points outside a circle of radius 3. The labels are not used for training, only to colour some plots.

**Evaluation protocol:** we hold out 20% of the training file (2,000 rows) as a **validation set**. It monitors training and sets the anomaly threshold. The test file is used **once**, at the end, to report how many rows the fixed threshold flags. The scaler is fitted on the training rows only.

**Learning objectives:**
1. Build an encoder-decoder architecture in PyTorch.
2. Train on reconstruction loss (MSE).
3. Visualize original vs. reconstructed samples.
4. Extract and visualize latent representations.
5. Use reconstruction error as an anomaly score, with a threshold set on validation data.

---
## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE

SEED = 42
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__}, device: {DEVICE}")

---
## 2. Load and Explore the Data

We explore the training file only; the test file is kept for the final check in section 8.

In [ ]:
df_train = pd.read_csv("classif_train.csv", index_col=0)
df_test  = pd.read_csv("classif_test.csv",  index_col=0)

print(f"Train shape: {df_train.shape}")
print(f"Test  shape: {df_test.shape}")
df_train.head()

In [ ]:
# Visualize the training file (the test file stays unseen until section 8)
fig, ax = plt.subplots(figsize=(7, 6))
scatter = ax.scatter(df_train["x"], df_train["y"], c=df_train["label"], cmap="viridis",
                     s=5, alpha=0.5)
ax.set_title("Training file")
ax.set_xlabel("x")
ax.set_ylabel("y")
plt.colorbar(scatter, ax=ax, label="label")
plt.tight_layout()
plt.show()

---
## 3. Prepare Data for PyTorch

We hold out 20% of the training file as validation rows (seeded split) **before** fitting the scaler, and fit the scaler on the remaining training rows only. The validation rows monitor training (section 5) and set the anomaly threshold (section 8); the test rows are transformed with the same scaler but not used until section 8.

For an autoencoder, the **target is the input itself**: `TensorDataset(X, X)`.

In [ ]:
feature_cols = ["x", "y"]
X_all = df_train[feature_cols].values.astype(np.float32)
labels_all = df_train["label"].values

# Hold out 20% of the training file for validation (monitoring and the anomaly threshold)
X_train, X_val, labels_train, labels_val = train_test_split(
    X_all, labels_all, test_size=0.2, random_state=SEED)
X_test = df_test[feature_cols].values.astype(np.float32)   # used once, section 8

scaler = StandardScaler().fit(X_train)                      # training rows only
X_train_sc = scaler.transform(X_train).astype(np.float32)
X_val_sc = scaler.transform(X_val).astype(np.float32)
X_test_sc = scaler.transform(X_test).astype(np.float32)

train_tensor = torch.tensor(X_train_sc)
val_tensor = torch.tensor(X_val_sc)
test_tensor = torch.tensor(X_test_sc)

BATCH_SIZE = 64
train_ds = TensorDataset(train_tensor, train_tensor)  # input = target for autoencoders
val_ds = TensorDataset(val_tensor, val_tensor)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

print(f"Training rows: {len(train_ds)}, validation rows: {len(val_ds)}, test rows: {len(test_tensor)}")
print(f"Feature dim: {X_train_sc.shape[1]}")

---
## 4. Build the Autoencoder

**Encoder:** input_dim -> 16 -> 8 -> latent_dim (here 1: a one-number bottleneck)
**Decoder:** latent_dim -> 8 -> 16 -> input_dim

In [ ]:
class Autoencoder(nn.Module):
    def __init__(self, input_dim=2, latent_dim=1):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Linear(8, latent_dim),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 8),
            nn.ReLU(),
            nn.Linear(8, 16),
            nn.ReLU(),
            nn.Linear(16, input_dim),
        )

    def forward(self, x):
        z = self.encoder(x)
        x_hat = self.decoder(z)
        return x_hat

    def encode(self, x):
        return self.encoder(x)


LATENT_DIM = 1
torch.manual_seed(SEED)
model = Autoencoder(input_dim=2, latent_dim=LATENT_DIM).to(DEVICE)
print(model)

---
## 5. Train the Autoencoder

The validation loss is computed on the held-out validation rows after every epoch; the test file is not used here. The training budget (100 epochs) is fixed in advance, and the validation curve is a diagnostic.

In [ ]:
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
NUM_EPOCHS = 100

train_losses = []
val_losses = []

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    running = 0.0
    for x_batch, _ in train_loader:
        x_batch = x_batch.to(DEVICE)
        optimizer.zero_grad()
        x_hat = model(x_batch)
        loss = criterion(x_hat, x_batch)
        loss.backward()
        optimizer.step()
        running += loss.item() * x_batch.size(0)
    train_losses.append(running / len(train_ds))

    model.eval()
    val_running = 0.0
    with torch.no_grad():
        for x_batch, _ in val_loader:          # validation rows, not the test file
            x_batch = x_batch.to(DEVICE)
            x_hat = model(x_batch)
            val_running += criterion(x_hat, x_batch).item() * x_batch.size(0)
    val_losses.append(val_running / len(val_ds))

    if epoch % 20 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d}/{NUM_EPOCHS}  "
              f"Train MSE: {train_losses[-1]:.6f}  "
              f"Val MSE: {val_losses[-1]:.6f}")

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(train_losses, label="Train MSE")
plt.plot(val_losses, label="Val MSE")
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.title("Reconstruction Loss")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 6. Visualize Original vs. Reconstructed Samples (validation rows)

In [ ]:
model.eval()
with torch.no_grad():
    X_val_hat = model(val_tensor.to(DEVICE)).cpu().numpy()

# Inverse transform back to original scale
X_val_orig = scaler.inverse_transform(X_val_sc)
X_val_recon = scaler.inverse_transform(X_val_hat)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(X_val_orig[:, 0], X_val_orig[:, 1],
                c=labels_val, cmap="viridis", s=10, alpha=0.6)
axes[0].set_title("Original Validation Data")
axes[0].set_xlabel("x")
axes[0].set_ylabel("y")

axes[1].scatter(X_val_recon[:, 0], X_val_recon[:, 1],
                c=labels_val, cmap="viridis", s=10, alpha=0.6)
axes[1].set_title("Reconstructed Validation Data")
axes[1].set_xlabel("x")
axes[1].set_ylabel("y")

plt.tight_layout()
plt.show()

---
## 7. Extract and Visualize Latent Representations

In [ ]:
model.eval()
with torch.no_grad():
    z_train = model.encode(train_tensor.to(DEVICE)).cpu().numpy()
    z_val = model.encode(val_tensor.to(DEVICE)).cpu().numpy()

print(f"Latent train shape:      {z_train.shape}")
print(f"Latent validation shape: {z_val.shape}")

In [ ]:
# For 1D latent space, plot histogram coloured by label
if LATENT_DIM == 1:
    fig, ax = plt.subplots(figsize=(8, 4))
    for lbl in np.unique(labels_train):
        mask = labels_train == lbl
        ax.hist(z_train[mask, 0], bins=50, alpha=0.5, label=f"Label {lbl}")
    ax.set_xlabel("Latent Value")
    ax.set_ylabel("Count")
    ax.set_title("Latent Space Distribution by Label")
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    # For higher-dim latent, use t-SNE to project to 2D
    z_2d = TSNE(n_components=2, random_state=42).fit_transform(z_train[:2000])
    plt.figure(figsize=(8, 6))
    plt.scatter(z_2d[:, 0], z_2d[:, 1], c=labels_train[:2000],
                cmap="viridis", s=5, alpha=0.5)
    plt.colorbar(label="Label")
    plt.title("t-SNE of Latent Space")
    plt.tight_layout()
    plt.show()

---
## 8. Anomaly Detection via Reconstruction Error

Points that the autoencoder struggles to reconstruct may be anomalies (far from the learned data manifold). The threshold is a **decision**: we take the 95th percentile of the reconstruction errors on the **validation** rows, so about 5% of normal-like rows will be flagged by construction. A percentile sets an alarm rate; it does not measure accuracy. We fix the threshold on the validation rows, then apply it **once** to the test file.

In [ ]:
# Reconstruction error (MSE per row) on the validation rows
model.eval()
with torch.no_grad():
    X_val_hat = model(val_tensor.to(DEVICE)).cpu().numpy()
val_error = np.mean((X_val_sc - X_val_hat) ** 2, axis=1)

# The threshold is chosen on the validation rows: their 95th percentile
threshold = np.percentile(val_error, 95)
print(f"Threshold (95th percentile of validation errors): {threshold:.6f}")

plt.figure(figsize=(8, 4))
plt.hist(val_error, bins=50, edgecolor="black", alpha=0.7)
plt.axvline(threshold, color="red", linestyle="--", label="Threshold (validation 95th percentile)")
plt.xlabel("Reconstruction Error (MSE)")
plt.ylabel("Count")
plt.title("Distribution of Reconstruction Errors (validation rows)")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Apply the fixed threshold once to the test file
with torch.no_grad():
    X_test_hat = model(test_tensor.to(DEVICE)).cpu().numpy()
test_error = np.mean((X_test_sc - X_test_hat) ** 2, axis=1)
anomalies = test_error > threshold
print(f"Test rows flagged: {anomalies.sum()} / {len(anomalies)} ({anomalies.mean():.1%}); "
      f"about 5% is expected if the test rows resemble the training rows")

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(X_test[~anomalies, 0], X_test[~anomalies, 1],
           c="steelblue", s=10, alpha=0.5, label="Below threshold")
ax.scatter(X_test[anomalies, 0], X_test[anomalies, 1],
           c="red", s=30, alpha=0.8, label="Flagged (above threshold)")
ax.set_title("Test Rows Flagged by the Validation Threshold")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.legend()
plt.tight_layout()
plt.show()

**Reading the result.** The flagged rows lie where the learned one-dimensional curve cannot reach, typically corners of the square and gaps between folds of the curve. That is a property of the bottleneck and the data geometry, not a physical anomaly, and the circle labels are unrelated to the reconstruction task. The flagged fraction on the test file is close to the 5% alarm rate chosen on validation, as expected when the test rows come from the same distribution. Extension: compare with a PCA reconstruction baseline (one component, fitted on the training rows).

---
## 9. Summary

In this notebook we:

1. Loaded a 2D dataset and visualized the training file.
2. Held out validation rows from the training file and fitted the scaler on the training rows only.
3. Built a simple **autoencoder** (encoder-decoder) in PyTorch and trained it on MSE reconstruction loss, monitoring the validation rows.
4. Visualized original vs. reconstructed validation points.
5. Extracted latent representations and visualized their distribution.
6. Set an anomaly threshold on the validation rows (95th percentile of their reconstruction errors) and applied it once to the test file.

**Key takeaways:**
- Autoencoders learn a compressed representation of the data without labels.
- The latent space captures the most important structure of the data.
- Reconstruction error is a natural anomaly score -- samples that are hard to reconstruct lie far from the learned manifold.
- A percentile threshold sets the alarm rate: choose it on validation data, report it on test data, and do not read a flag as a diagnosis.